In [33]:
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'
#pd.set_option('display.width', 1000)  # display all columns without wrapping
#pd.set_option('display.max_columns', None)  # display all columns
#pd.set_option('display.max_rows', None)  # display all rows
pd.set_option('display.expand_frame_repr', False) # disable wrapping

In [34]:
import yfinance as yf
from concurrent.futures import ThreadPoolExecutor

def get_ext_price(symbol: str, include_extended_hours: bool = True) -> float:
	ticker = yf.Ticker(symbol)

	if include_extended_hours:
		info = ticker.info or {}
		# Prefer explicit post-market and pre-market quotes when available.
		post_market_price = info.get("postMarketPrice")
		if post_market_price is not None:
			return round(float(post_market_price), 2)

		pre_market_price = info.get("preMarketPrice")
		if pre_market_price is not None:
			return round(float(pre_market_price), 2)

		# Fallback for extended hours: latest intraday trade including pre/post data.
		intraday = ticker.history(period="1d", interval="1m", prepost=True)
		if not intraday.empty:
			close_series = intraday["Close"].dropna()
			if not close_series.empty:
				return round(float(close_series.iloc[-1]), 2)

	# Try the most up-to-date value first.
	fast_info = ticker.fast_info or {}
	last_price = fast_info.get("lastPrice")
	if last_price is not None:
		return round(float(last_price), 2)

	# Fallback: use latest close from recent history.
	history = ticker.history(period="5d")
	if history.empty:
		raise RuntimeError(f"Unable to fetch {symbol} price from yfinance")
	return round(float(history["Close"].iloc[-1]), 2)

def get_prices(tickers, include_extended_hours: bool = True) -> dict:
	"""Fetch prices for unique tickers concurrently; failures map to NaN."""
	def safe_get(symbol):
		try:
			return get_ext_price(symbol, include_extended_hours)
		except Exception as e:
			print(symbol, e)
			return float('nan')
	unique = list(dict.fromkeys(tickers))
	with ThreadPoolExecutor(max_workers=16) as ex:
		prices = list(ex.map(safe_get, unique))
	return dict(zip(unique, prices))
#print(get_ext_price("AAPL")) #testing   

In [35]:
df = pd.read_csv(r'G:\My Drive\tax\out_fidelity.csv', on_bad_lines='skip')

df = df[df['Option'].isna() & ~df['Ticker'].isin(['sell', 'buy'])] # .head(10) Limit to 10 rows for testing, rows Option column empty and not buy or sell
#print(df[['Ticker', 'Option','OptionCnt', 'sell', 'CostBasisPerShare','buy']]) #testing
df['o_CurPrice'] = df['Ticker'].map(get_prices(df['Ticker'], include_extended_hours=False)) # fast_info only, each unique ticker once, in parallel
df['o_ref_price'] = df['sell'].combine_first(pd.to_numeric(df['CostBasisPerShare'], errors='coerce').abs())
#print(df[['Ticker', 'sell', 'CostBasisPerShare','o_CurPrice', 'o_ref_price']]) #testing

df['o_ref_price'] = pd.to_numeric(df['o_ref_price'], errors='coerce')
df['o_Diff'] = df['o_CurPrice'] - df['o_ref_price']

df = df.sort_values('o_Diff', ascending=False)
df['sell'] = df['sell'].fillna('')
df['OptionCnt'] = df['OptionCnt'].apply(lambda x: int(x) if pd.notna(x) else '')
print(df[['Account', 'Ticker', 'CostBasisPerShare', 'sell', 'o_CurPrice', 'o_Diff', 'OptionCnt']])

         Account Ticker CostBasisPerShare   sell  o_CurPrice  o_Diff  OptionCnt
41     218320886   CRCL           -182.66   90.0      101.09   11.09          1
62     wRoth8472   ETHE             -26.7   21.0       19.65   -1.35          1
74     xRoth8929    MNY            -14.15    2.5        0.86   -1.64          1
58     233186075    DJT            -24.79   11.0        9.02   -1.98          2
86     231736622   NNOX             -9.21    3.1        0.76   -2.34          3
83     233186075   NNOX             -45.7    3.5        0.76   -2.74          1
84     218320886   NNOX            -15.83    3.5        0.76   -2.74          4
85     414745529   NNOX             -13.6    3.5        0.76   -2.74          1
134    218320886   TLRY           -126.98    8.0        4.49   -3.51          1
93     218320886   PPLT            -23.57   20.0       16.43   -3.57          9
128    233186075   SQQQ           -111.45   45.0       38.51   -6.49          1
97     218320886   RGTI             -25.

In [36]:
import datetime as dt
import re
from datetime import date
from concurrent.futures import ThreadPoolExecutor
import yfinance as yf

def calc_rate(cost, symbol_to):
    if not symbol_to.strip():
        return
    symbolOp = re.split(r'([\d.]+)', symbol_to)
    final_value = float(symbolOp[3])
    final_date = dt.datetime.strptime(str(int(symbolOp[1])), '%y%m%d').date()
    days_hold = (final_date - date.today()).days+1
    rate_of_gain = float(cost) / final_value * (365 / days_hold) * 100
    return round(rate_of_gain)

def _fetch_chains(keys):
    """Fetch option chains for unique (ticker, expiry) pairs concurrently."""
    def fetch(key):
        try:
            return key, yf.Ticker(key[0]).option_chain(key[1])
        except Exception as e:
            print(key[0], e)
            return key, None
    with ThreadPoolExecutor(max_workers=16) as ex:
        return dict(ex.map(fetch, keys))

def _scan(df, opt_type):
    # Parse all symbols first so each (ticker, expiry) chain is downloaded only once.
    parsed = {}
    for i in df.index:
        try:
            symbolOp = re.split(r'([\d.]+)', df['Symbol'][i])
            myDate = dt.datetime.strptime(str(int(symbolOp[1])), '%y%m%d').date()
            parsed[i] = (symbolOp[0], myDate.strftime('%Y-%m-%d'), float(symbolOp[3]))
        except Exception as e:
            print(df['Symbol'][i], e)
    chains = _fetch_chains({(tkr, expiry) for tkr, expiry, _ in parsed.values()})
    for i, (tkr, expiry, strike) in parsed.items():
        try:
            optC = chains[(tkr, expiry)]
            opt = optC.puts if opt_type == 'P' else optC.calls
            opt = opt[ (opt['strike'] - strike).abs() < 1 ]
            df.loc[i,'o_Buy'] = pd.Series(opt['bid']).values[0]
            sell = float(df['sell'][i])
            df.loc[i,'o_GainLoss'] = (df['o_Buy'][i] - sell) * df['Quantity'][i]
            df.loc[i,'o_percent'] = round(df['o_GainLoss'][i] * 100 / sell / abs(df['Quantity'][i]))
            df.loc[i, 'o_rate'] = calc_rate(df['o_Buy'][i], df['Symbol'][i])
        except Exception as e:
            print(tkr, e)
    df['o_percent'] = pd.to_numeric(df['o_percent'], errors='coerce').astype('Int64')
    df['o_rate'] = pd.to_numeric(df['o_rate'], errors='coerce').astype('Int64')
    return df[['Account', 'Symbol', 'o_Buy', 'sell', 'o_GainLoss', 'o_percent', 'o_rate']]

In [37]:
def _scan_P(df):
    return _scan(df, 'P')

In [38]:
pd.options.mode.chained_assignment = None  # default='warn'
pd.set_option('display.width', 1000)  # display all columns without wrapping
pd.set_option('display.max_columns', None)  # display all columns
pd.set_option('display.max_rows', None)  # display all rows

df = pd.read_csv(r'G:\My Drive\tax\out_fidelity.csv', on_bad_lines='skip')
df = df[(df['Option'] == 'P') & (~df['Ticker'].isin(['sell', 'buy']))]
#print(df[['Account', 'Symbol', 'sell']]) #testing
df_P = _scan_P(df)
df_P = df_P.sort_values(by=['o_percent'], ascending=False)
print(df_P)

         Account            Symbol  o_Buy    sell  o_GainLoss  o_percent  o_rate
107    218320886     SMCI260904P37   0.00    1.25        1.25        100       0
44     218320886     CRCL260904P94   0.00    4.10        4.10        100       0
14     X83939133     AMD260904P475   2.44   14.50       12.06         83     187
94     218320886     PPLT260918P16   0.10    0.45        0.35         78      15
112    218320886   SNDK260918P1560  34.40  148.60      114.20         77      54
119    X83939133    SPCX260911P140   0.97    3.90        2.93         75      32
132    218320886      TEM260911P61   0.77    2.55        1.78         70      58
15     X83939133     APP260918P310   6.90   21.30       14.40         68      54
30     218320886    COIN260911P175   2.05    6.20        4.15         67      53
98     218320886     RGTI260904P16   0.30    0.50        0.20         40     684
24     218320886    CBRS260918P220  15.70   25.00        9.30         37     174
104    X65750304      SLV260

In [39]:
def _scan_C(df):
    return _scan(df, 'C')

In [40]:
pd.set_option('display.width', 1000)  # display all columns without wrapping
df = pd.read_csv(r'G:\My Drive\tax\out_fidelity.csv', on_bad_lines='skip')
df = df[(df['Option'] == 'C') & (~df['Ticker'].isin(['sell', 'buy']))]
# #df = df[~df['Symbol'].str.contains('260515')] # filter rows, keep only current month call 
# #df = df[df['Account'] == 'X65750304'] # only rows on account   
df_C = _scan_C(df)
df_C = df_C.sort_values(by=['o_percent'], ascending=False)
print(df_C)
# #df_C.reset_index(drop=True)

         Account            Symbol   o_Buy   sell  o_GainLoss  o_percent  o_rate
70     414745529     IONQ260904C42    0.00   2.05        2.05        100       0
66     X65750304     GLD260918C429    1.59  13.00       11.41         88       9
137    X83939133  TSLA260904C352.5    1.34   8.90        7.56         85     139
12     218320886      AGQ260918C90    2.20   9.10       13.80         76      59
140    X83939133  TSLA260911C367.5    2.76  10.50        7.74         74      34
11     218320886      AGQ260918C86    2.70   8.50        5.80         68      76
32     218320886  COIN260911C187.5    4.90  14.80        9.90         67     119
125    218320886    SPCX260918C150    5.20  13.30        8.10         61      84
60     233186075      DJT261016C10    0.49   1.15        0.66         57      42
10     X83939133      AGQ260918C82    4.40   9.30        9.80         53     131
39     X65750304  COIN260918C192.5    6.10  12.70        6.60         52      77
31     218320886    COIN2609